# 🧪 CDN Module Resolution Test

This notebook validates that the `@dyne/interfacer-client` and `zenroom` packages can be loaded in the JupyterLite JavaScript kernel via CDN.

## 🆚 CDN vs NPM: Why we use ESM imports here

| | CDN (JupyterLite) | NPM (Node.js) |
|---|---|---|
| **Environment** | Browser Web Worker | Node.js / server |
| **Install** | `import('https://esm.sh/pkg')` | `pnpm add pkg` |
| **Filesystem** | None (in-memory) | `node_modules/` |
| **Build step** | None (on-the-fly) | Bundler (tsup, webpack) |
| **Use case** | Learning, prototyping | Production apps |

**esm.sh** is a CDN that takes any npm package and bundles it for browsers as an ES module.
It resolves `package.json` exports, transpiles TypeScript, and handles CJS→ESM conversion.

> 💡 **In production**, you would use:
> ```bash
> pnpm add @dyne/interfacer-client zenroom
> ```
> ```js
> import { InterfacerClient } from '@dyne/interfacer-client';
> ```

## 🔑 Key limitation

**Zenroom** (the WASM crypto VM) may not work in all browser environments.
It requires SharedArrayBuffer and COOP/COEP headers, which JupyterLite may not provide.
We test this below — if it fails, auth-dependent features won't work in-browser.

## 1. Load the sandbox configuration

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  // Load sandbox config from the static file
  import { SANDBOX_CONFIG, SANDBOX_ENDPOINTS } from './config.js';
  
  console.log('Sandbox proxy:', SANDBOX_CONFIG.proxyUrl);
  console.log('Zenflows URL:', SANDBOX_ENDPOINTS.zenflowsUrl);
  console.log('DPP URL:', SANDBOX_ENDPOINTS.dppUrl);
})();


## 2. Import InterfacerClient from CDN

In [ ]:
(async () => {
  // Dynamic import from esm.sh CDN
  const { InterfacerClient } = await import('https://esm.sh/@dyne/interfacer-client');
  
  console.log('InterfacerClient imported successfully:', typeof InterfacerClient);  
  // Expose to other cells via window
  if (!window._if) window._if = {};
  window._if.InterfacerClient = InterfacerClient;
  window._if.SANDBOX_CONFIG = SANDBOX_CONFIG;
  window._if.client = client;
})();


## 3. Instantiate the client with sandbox config

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  const client = new InterfacerClient(SANDBOX_CONFIG);
  
  console.log('Client instantiated.');
  console.log('Config:', client.config);
  console.log('Sub-clients:', {
    auth: !!client.auth,
    graphql: !!client.graphql,
    resources: !!client.resources,
    files: !!client.files,
    dpp: !!client.dpp,
    inbox: !!client.inbox,
    wallet: !!client.wallet,
    social: !!client.social,
    tagging: !!client.tagging,
    import: !!client.import,
  });
})();


## 4. Test Zenroom WASM import

In [ ]:
(async () => {
  // Get shared state
  const _if = window._if || {};
  const InterfacerClient = _if.InterfacerClient;
  const SANDBOX_CONFIG = _if.SANDBOX_CONFIG;
  const client = _if.client;
  // Zenroom is a peer dependency of interfacer-client
  try {
    const zenroom = await import('https://esm.sh/zenroom');
    console.log('Zenroom imported:', typeof zenroom.default || typeof zenroom);
  } catch (err) {
    console.warn('Zenroom import failed (expected in browser without WASM support):', err.message);
  }
})();


## ✅ Test Results

If all cells above executed without errors, the CDN import pipeline is working correctly.

**Next step:** Move on to `00_Quick_Start.ipynb` for a full walkthrough.